# Projekt 01 (basic) — Das Explore-Exploit-Dilemma: Multi-armed Bandits

> **Modul 13 — Reinforcement Learning** · Format: **Jupyter Notebook**
>
> **Warum dieses Format?** Bandits sind der einfachste RL-Fall (ein Zustand) und isolieren die
> *eine* Frage, die RL von allem anderen unterscheidet: **explorieren oder exploitieren?**
> Ein Notebook ist ideal, weil hier Code, kurze Experimente und vor allem **Lernkurven-Plots**
> zusammengehören — man *sieht* den Unterschied der Strategien direkt.

## Ziel
Du baust das **k-armed Testbed** von Sutton & Barto und vier Aktions-Auswahl-Strategien und
vergleichst sie empirisch:
- **greedy** (immer die bisher beste Aktion — reines Exploit),
- **ε-greedy** (mit Wahrscheinlichkeit ε zufällig),
- **optimistische Initialisierung** (hohe Startwerte erzwingen frühe Exploration),
- **UCB** (Upper Confidence Bound — „Optimismus bei Unsicherheit").

## Vorwissen
Skript Modul 13, Abschnitt **3.1** (Explore-Exploit, Bandits). Python/NumPy-Grundlagen.

## Was am Ende funktionieren soll
Zwei Plots wie im Sutton-&-Barto-Kapitel 2: **mittlere Belohnung** und **% optimale Aktion**
über die Zeit, gemittelt über viele zufällige Bandit-Probleme — plus dein Fazit, welche
Strategie wann gewinnt.

> **Arbeitsweise:** Versuch die mit `# TODO` markierten Stellen zuerst **selbst**. Die
> vollständige Musterlösung liegt in `loesung/bandits_loesung.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng_global = np.random.default_rng(0)
print("numpy", np.__version__)

## 1 · Die Umgebung: das k-armed Gaussian Testbed

Ein Bandit hat $k$ Arme. Für jeden Arm $a$ gibt es einen **wahren Wert** $q_*(a)$, gezogen aus
$\mathcal N(0,1)$. Ziehst du Arm $a$, bekommst du eine **verrauschte** Belohnung
$R \sim \mathcal N(q_*(a), 1)$. Der Agent kennt $q_*$ **nicht** und muss es aus den beobachteten
Belohnungen schätzen. Die **optimale** Aktion ist $a^* = \arg\max_a q_*(a)$.

Diese Klasse ist vollständig vorgegeben — sie ist nur die *Umgebung*, nicht der Lernteil.

In [ ]:
class GaussianBandit:
    # k-armed Bandit: q*(a) ~ N(0,1), Belohnung R ~ N(q*(a), 1).
    def __init__(self, k=10, seed=None):
        self.k = k
        self.rng = np.random.default_rng(seed)
        self.q_true = self.rng.normal(0.0, 1.0, size=k)   # wahre Werte q*(a)
        self.optimal_action = int(np.argmax(self.q_true))

    def step(self, a):
        # Ziehe Arm a, gib eine verrauschte Belohnung zurueck.
        return self.rng.normal(self.q_true[a], 1.0)

# kleine Demo: ein Bandit, 5 Zuege auf Arm 0
demo = GaussianBandit(k=10, seed=42)
print("wahre Werte q*(a):", np.round(demo.q_true, 2))
print("optimaler Arm:", demo.optimal_action, " q* =", round(demo.q_true[demo.optimal_action], 2))
print("5 Belohnungen von Arm 0:", np.round([demo.step(0) for _ in range(5)], 2))

## 2 · Der Agent

Der Agent schätzt für jeden Arm einen Wert $Q(a)$ und zählt die Züge $N(a)$. Nach jedem Zug
aktualisiert er inkrementell (**sample average**):
$$N(a)\leftarrow N(a)+1,\qquad Q(a)\leftarrow Q(a)+\tfrac1{N(a)}\big(R - Q(a)\big).$$

Die **Auswahl** hängt von der Strategie ab:
- `greedy` / `epsilon`: mit Wahrscheinlichkeit $\varepsilon$ ein zufälliger Arm, sonst
  $\arg\max_a Q(a)$ (greedy = ε=0).
- `ucb`: wähle $\arg\max_a\big[Q(a) + c\sqrt{\ln t / N(a)}\big]$; ein noch nie gezogener Arm
  ($N(a)=0$) hat unendlichen Bonus, wird also **zuerst** probiert.
- Die **optimistische Initialisierung** ist keine eigene Auswahl-Regel, sondern nur ein hoher
  Startwert `Q_init` bei ansonsten greedy-Auswahl.

**Deine Aufgabe:** Fülle `select_action` und `update` aus.

In [ ]:
class BanditAgent:
    def __init__(self, k=10, strategy="epsilon", epsilon=0.1, c=2.0, Q_init=0.0):
        self.k = k
        self.strategy = strategy      # "greedy" | "epsilon" | "ucb"
        self.epsilon = epsilon
        self.c = c
        self.Q = np.full(k, float(Q_init))   # Wertschaetzungen (optimistisch, wenn Q_init hoch)
        self.N = np.zeros(k, dtype=int)       # Anzahl Zuege pro Arm
        self.t = 0                            # Gesamt-Zeitschritt
        self.rng = np.random.default_rng()

    def select_action(self):
        self.t += 1
        # TODO: Auswahl je nach self.strategy implementieren.
        #  - "epsilon": mit Wkt. self.epsilon self.rng.integers(self.k),
        #               sonst argmax(self.Q). ("greedy" = epsilon 0.)
        #  - "ucb": bevorzuge zuerst Arme mit N==0 (unendlicher Bonus);
        #           sonst argmax(self.Q + self.c*sqrt(ln(self.t)/self.N)).
        # Tipp bei Gleichstand: np.argmax nimmt den ersten -> fuer faire
        #   Zufallswahl bei Ties kannst du np.flatnonzero(x==x.max()) + rng.choice nutzen.
        raise NotImplementedError

    def update(self, a, r):
        # TODO: inkrementelles sample-average-Update fuer Arm a mit Belohnung r.
        raise NotImplementedError


## 3 · Das Experiment-Harness

Ein einzelner Bandit ist zu verrauscht, um Strategien zu vergleichen. Deshalb der **Testbed**:
mittle über **viele** zufällige Bandit-Probleme (`n_runs`), jeweils über `n_steps` Züge. Wir
protokollieren pro Schritt (a) die erhaltene **Belohnung** und (b) ob die **optimale Aktion**
gewählt wurde. Vorgegeben — hier ist nichts zu tun.

In [ ]:
def run_experiment(agent_kwargs, k=10, n_steps=1000, n_runs=2000, seed=0):
    seeder = np.random.default_rng(seed)
    rewards = np.zeros((n_runs, n_steps))
    optimal = np.zeros((n_runs, n_steps))
    for run in range(n_runs):
        env = GaussianBandit(k=k, seed=int(seeder.integers(1 << 30)))
        agent = BanditAgent(k=k, **agent_kwargs)
        agent.rng = np.random.default_rng(int(seeder.integers(1 << 30)))
        for t in range(n_steps):
            a = agent.select_action()
            r = env.step(a)
            agent.update(a, r)
            rewards[run, t] = r
            optimal[run, t] = 1.0 if a == env.optimal_action else 0.0
    return rewards.mean(axis=0), optimal.mean(axis=0)

# Schnell-Check mit wenigen Runs (das grosse Experiment kommt danach)
avg_r, avg_opt = run_experiment(dict(strategy="epsilon", epsilon=0.1), n_steps=200, n_runs=200)
print("nach 200 Schritten:  mittl. Belohnung =", round(avg_r[-1], 3),
      "| %optimal =", round(100*avg_opt[-1], 1))

## 4 · Der große Vergleich

Wir vergleichen fünf Konfigurationen (die klassischen Sutton-&-Barto-Kurven):

| Label | strategy | Parameter |
|---|---|---|
| greedy | `greedy` | — |
| ε=0.1 | `epsilon` | ε=0.1 |
| ε=0.01 | `epsilon` | ε=0.01 |
| optimistisch, greedy | `greedy` | Q_init=5 |
| UCB c=2 | `ucb` | c=2 |

> Das volle Experiment (2000 Runs × 1000 Schritte × 5 Configs) dauert auf der CPU **~1–2 min**.
> Zum schnellen Testen kannst du `N_RUNS`/`N_STEPS` reduzieren.

In [ ]:
N_STEPS, N_RUNS = 1000, 2000   # bei Bedarf kleiner setzen, z.B. 500/500

configs = {
    "greedy":               dict(strategy="greedy"),
    "$\\varepsilon$=0.1":   dict(strategy="epsilon", epsilon=0.1),
    "$\\varepsilon$=0.01":  dict(strategy="epsilon", epsilon=0.01),
    "optimistisch, greedy": dict(strategy="greedy", Q_init=5.0),
    "UCB c=2":              dict(strategy="ucb", c=2.0),
}

results = {}
for i, (label, kw) in enumerate(configs.items()):
    results[label] = run_experiment(kw, k=10, n_steps=N_STEPS, n_runs=N_RUNS, seed=100 + i)
    print(f"fertig: {label:22s}  %optimal(final) = {100*results[label][1][-1]:.1f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for label, (r, o) in results.items():
    ax1.plot(r, label=label, lw=1.3)
    ax2.plot(100 * o, label=label, lw=1.3)
ax1.set(xlabel="Schritt", ylabel="mittlere Belohnung", title="Mittlere Belohnung")
ax2.set(xlabel="Schritt", ylabel="% optimale Aktion", title="% optimale Aktion", ylim=(0, 100))
for ax in (ax1, ax2):
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5 · Regret

Der **Regret** ist die aufsummierte *entgangene* Belohnung gegenüber dem immer-optimalen Spiel:
$$\rho_T = \sum_{t=1}^{T}\big(q_*(a^*) - q_*(A_t)\big).$$
Eine gute Strategie hat **sublinearen** (idealerweise logarithmischen) Regret — die Kurve flacht
ab. Wir approximieren $q_*(a^*)-q_*(A_t)$ über den Testbed durch den mittleren Belohnungs-Rückstand
zum theoretischen Optimum (Erwartungswert des Maximums von 10 Std-Normalen ≈ 1.54).

In [ ]:
# E[max von 10 N(0,1)] ~ 1.5388 (Monte-Carlo-Schaetzung des Testbed-Optimums)
q_star_max = np.mean([GaussianBandit(k=10, seed=s).q_true.max() for s in range(5000)])
print("mittleres Optimum q*(a*) ~", round(q_star_max, 3))

plt.figure(figsize=(7, 4.5))
for label, (r, o) in results.items():
    regret = np.cumsum(q_star_max - r)
    plt.plot(regret, label=label, lw=1.4)
plt.xlabel("Schritt"); plt.ylabel("kumulierter Regret")
plt.title("Regret (niedriger = besser; flach = sublinear)")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6 · Beobachtungen & Fazit

Trage deine Beobachtungen ein (in der Lösung ausgefüllt). Typisch für den 10-armed Testbed:

- **greedy** steigt schnell, bleibt aber früh stecken (~ nur um 30–40 % optimale Aktion) — es
  fixiert sich auf einen Arm, der zufällig gut aussah, und exploriert nie mehr. **Reiner Exploit
  verliert.**
- **ε=0.1** exploriert kräftig → findet den besten Arm schnell, aber die Dauer-Exploration
  deckelt es bei ~91 % optimaler Aktion. **ε=0.01** ist langsamer, aber langfristig besser
  (weniger „Störfeuer"). → **Explore-Exploit-Trade-off in einem Bild.**
- **optimistische Initialisierung** (Q₀=5) exploriert *von selbst* am Anfang (jeder ungezogene
  Arm wirkt zu gut) → charakteristischer *Buckel* früh, dann sehr gut — aber nur ein
  **Anfangs**-Trick (nutzlos bei nichtstationären Problemen).
- **UCB** ist hier meist der Gesamtsieger: gezielte Exploration nach Unsicherheit statt blindem
  Zufall → schnell **und** hoch, mit dem flachsten Regret.

**Die Kernbotschaft:** Es gibt kein „richtiges" ε — jede Strategie ist ein anderer Punkt im
Explore-Exploit-Trade-off. Genau dieses Dilemma kehrt im vollen RL (nächste Projekte) wieder,
nur mit **vielen Zuständen**.

### Mini-Aufgaben zum Weiterprobieren
1. Setze das UCB-`c` auf 0.5 und 4 — wie ändert sich die Kurve?
2. Baue eine **nichtstationäre** Variante: lasse `q_true` pro Schritt leicht driften
   (`q_true += rng.normal(0, 0.01, k)`). Warum schlägt jetzt ein **konstantes** Schritt-Update
   $Q\leftarrow Q+\alpha(R-Q)$ das sample-average? (→ Skript 2.2: „vergisst alte Erfahrung").
3. Ergänze eine **Softmax/Boltzmann**-Auswahl ($\propto e^{Q(a)/\tau}$) und vergleiche.